Run `python -m src.pipeline` first, it creates `data/output/tech_news.duckdb`.

In [1]:
from pathlib import Path
import duckdb

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

con = duckdb.connect(str(ROOT / "data/output/tech_news.duckdb"), read_only=True)

In [2]:
con.sql("SHOW TABLES").df()

,name
0,dim_company
1,fact_arr_observation
2,fact_article
3,fact_article_embedding


In [3]:
con.sql("SELECT * FROM dim_company").df()

,company_id,company_name,industry,founded_year,headquarters,employee_count,is_public,stock_ticker,company_size_category,in_metadata
0,airbnb,Airbnb,Data Analytics,1999,"London, UK",19967,False,NaN,Medium,True
1,amazon_web_services,Amazon Web Services,SaaS,2015,"Berlin, Germany",27826,False,NaN,Medium,True
2,anthropic,Anthropic,FinTech,2006,"San Francisco, CA",43747,False,NaN,Large,True
3,cloudflare,Cloudflare,Cybersecurity,1998,"Austin, TX",4422,False,CLOU,Small,True
4,cohere,Cohere,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,False
5,confluent,Confluent,Cloud Computing,1995,"London, UK",3884,False,NaN,Small,True
6,databricks,Databricks,Data Analytics,2003,"Berlin, Germany",17962,True,DATA,Medium,True
7,datarobot,DataRobot,Cloud Computing,2012,"New York, NY",23471,True,NaN,Medium,True
8,elastic,Elastic,Cybersecurity,2016,"San Francisco, CA",24508,True,ELAS,Medium,True
9,google_deepmind,Google DeepMind,Data Analytics,2017,"Austin, TX",24274,False,NaN,Medium,True


In [4]:
# Rows per table
con.sql("""
    SELECT 'dim_company' AS table_name, COUNT(*) AS row_count FROM dim_company
    UNION ALL SELECT 'fact_article', COUNT(*) FROM fact_article
    UNION ALL SELECT 'fact_arr_observation', COUNT(*) FROM fact_arr_observation
""").df()

,table_name,row_count
0,dim_company,26
1,fact_article,750
2,fact_arr_observation,558


In [5]:
# Exploration of articles by company
con.sql("""
    SELECT com.company_name,
        COUNT(*) as article_count,
        MIN(obs.as_of_date) as first_article_date,
        MAX(obs.as_of_date) as last_article_date,
        MIN(obs.arr_usd) as min_arr_usd,
        MAX(obs.arr_usd) as max_arr_usd
    FROM fact_arr_observation obs
    INNER JOIN dim_company com ON com.company_id = obs.company_id
    GROUP BY com.company_name
    ORDER BY article_count DESC
""").df()

,company_name,article_count,first_article_date,last_article_date,min_arr_usd,max_arr_usd
0,Confluent,33,2020-01-15,2024-10-27,260000000,1156000000
1,MongoDB,31,2020-02-29,2024-11-13,500000000,2057000000
2,Microsoft,30,2020-08-06,2024-12-02,44500000000,83850000000
3,SpaceX,29,2020-06-11,2024-12-12,2100000000,9945000000
4,Scale AI,29,2020-01-24,2024-10-10,100000000,1307000000
5,Cloudflare,28,2020-01-14,2024-12-23,390000000,1647000000
6,Meta AI,27,2020-04-20,2024-12-19,130000000,1950000000
7,Stripe,27,2020-01-16,2024-12-02,720000000,3910000001
8,Google DeepMind,26,2020-04-24,2024-12-21,120000000,1215000000
9,DataRobot,26,2020-05-20,2024-12-12,150000000,407000000


In [6]:
# Latest ARR of each company
con.sql("""
    SELECT * FROM fact_arr_observation
    WHERE company_id = ?
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY company_id
        ORDER BY as_of_date DESC, article_id DESC
    ) = 1
""", params=["openai"]).df()

,article_id,company_id,as_of_date,arr_usd,arr_raw
0,ART0710,openai,2024-10-28,4050000000,$4.050B


In [7]:
# Last Quarter ARR by company and year
con.sql("""
    SELECT *, YEAR(as_of_date) AS year, QUARTER(as_of_date) AS quarter
    FROM fact_arr_observation
    WHERE company_id = ?
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY company_id, YEAR(as_of_date), QUARTER(as_of_date)
        ORDER BY as_of_date DESC, article_id DESC
    ) = 1    
    ORDER BY year, quarter
""", params=["openai"]).df()

,article_id,company_id,as_of_date,arr_usd,arr_raw,year,quarter
0,ART0102,openai,2020-06-08,75000000,$75.0M,2020,2
1,ART0155,openai,2020-11-19,95000000,$0.095 billion,2020,4
2,ART0390,openai,2021-03-30,130000000,"¥19,500,000,000",2021,1
3,ART0403,openai,2021-05-21,180000000,"€163,636,364",2021,2
4,ART0188,openai,2021-07-13,240000000,240.0M USD,2021,3
5,ART0011,openai,2021-10-25,320000000,$320.0M,2021,4
6,ART0639,openai,2023-02-07,1450000000,$1450.0 million,2023,1
7,ART0634,openai,2023-06-10,1600000000,1600.0M USD,2023,2
8,ART0402,openai,2023-08-22,1720000000,$1720.0 million,2023,3
9,ART0549,openai,2023-10-21,1800000000,$1800.0 million,2023,4


In [8]:
# Status of the extracted ARR by article
con.sql("""
    SELECT COALESCE(arr_exclusion_reason, '(valid ARR)') AS reason, COUNT(*) AS articles
    FROM fact_article
    GROUP BY reason
    ORDER BY articles DESC
""").df()

,reason,articles
0,(valid ARR),558
1,undisclosed,126
2,missing,66


In [9]:
# Companies that are not in company_metadata.json (no enriched data: industry, founded year, etc...)
con.sql("""
    SELECT com.company_id, com.company_name, COUNT(art.article_id) AS articles
    FROM dim_company com
    LEFT JOIN fact_article art ON art.company_id = com.company_id
    WHERE com.in_metadata = False
    GROUP BY com.company_id, com.company_name
    ORDER BY articles DESC
""").df()

,company_id,company_name,articles
0,cohere,Cohere,8
1,xai,xAI,6
2,mistral_ai,Mistral AI,5
3,hugging_face,Hugging Face,3
4,perplexity_ai,Perplexity AI,3
